In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import skew

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!ls -l "/content/drive/MyDrive/alrIEEEna_26_dataset/MLChallengeDataset/"

train = pd.read_csv("/content/drive/MyDrive/alrIEEEna_26_dataset/MLChallengeDataset/TRAIN.csv")

test = pd.read_csv("/content/drive/MyDrive/alrIEEEna_26_dataset/MLChallengeDataset/TEST.csv")


Mounted at /content/drive
total 22878
-rw------- 1 root root     2076 Mar  3 08:35 readme.txt
-rw------- 1 root root  4711205 Mar  3 08:35 TEST.csv
-rw------- 1 root root 18712865 Mar  3 08:35 TRAIN.csv


In [3]:
train = train.drop_duplicates().reset_index(drop=True)

In [4]:
X = train.drop("Class", axis=1)
y = train["Class"]
X_test = test.drop("ID", axis=1)
test_ids = test["ID"]

In [5]:
def engineer_features(df):
    f_cols = [c for c in df.columns if c.startswith('F')]
    df['row_mean'] = df[f_cols].mean(axis=1)
    df['row_std']  = df[f_cols].std(axis=1)
    df['row_max']  = df[f_cols].max(axis=1)
    df['row_min']  = df[f_cols].min(axis=1)
    df['row_skew'] = df[f_cols].apply(lambda x: skew(x), axis=1)

    top_correlates = ['F01', 'F09', 'F19', 'F29']
    for i in range(len(top_correlates)):
        for j in range(i + 1, len(top_correlates)):
            c1, c2 = top_correlates[i], top_correlates[j]
            df[f'{c1}_{c2}_mul'] = df[c1] * df[c2]
            df[f'{c1}_{c2}_div'] = df[c1] / (df[c2] + 1e-6)

    return df

In [6]:

X = engineer_features(X)
X_test = engineer_features(X_test)

In [7]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [8]:
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_mlp = np.zeros(len(X))

In [9]:
test_lgb = np.zeros(len(X_test))
test_xgb = np.zeros(len(X_test))
test_mlp = np.zeros(len(X_test))

In [11]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Processing Fold {fold + 1} / {n_splits} ---")
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)
    X_te_s  = scaler.transform(X_test)
    lgb_m = lgb.LGBMClassifier(
        n_estimators=3000, learning_rate=0.02, num_leaves=64,
        class_weight='balanced', random_state=42, importance_type='gain'
    )
    lgb_m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    xgb_m = xgb.XGBClassifier(
        n_estimators=3000, learning_rate=0.03, max_depth=6,
        eval_metric="aucpr", scale_pos_weight=1.5, random_state=42
    )
    xgb_m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    mlp_m = MLPClassifier(
        hidden_layer_sizes=(256, 128, 64), activation='relu',
        solver='adam', max_iter=300, random_state=42
    )
    mlp_m.fit(X_tr_s, y_tr)
    oof_lgb[val_idx] = lgb_m.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx] = xgb_m.predict_proba(X_val)[:, 1]
    oof_mlp[val_idx] = mlp_m.predict_proba(X_val_s)[:, 1]
    test_lgb += lgb_m.predict_proba(X_test)[:, 1] / n_splits
    test_xgb += xgb_m.predict_proba(X_test)[:, 1] / n_splits
    test_mlp += mlp_m.predict_proba(X_te_s)[:, 1] / n_splits


--- Processing Fold 1 / 5 ---
[LightGBM] [Info] Number of positive: 13849, number of negative: 20581
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.044077 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16320
[LightGBM] [Info] Number of data points in the train set: 34430, number of used features: 64
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1625]	valid_0's binary_logloss: 0.0290445

--- Processing Fold 2 / 5 ---
[LightGBM] [Info] Number of positive: 13849, number of negative: 20581
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024143 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 16320
[LightGBM] [Info] Number of data poin

In [12]:
print("\nTraining Meta-Model (Logistic Regression)...")
X_meta = np.column_stack([oof_lgb, oof_xgb, oof_mlp])
X_meta_test = np.column_stack([test_lgb, test_xgb, test_mlp])

meta_model = LogisticRegression()
meta_model.fit(X_meta, y)

final_oof_probs = meta_model.predict_proba(X_meta)[:, 1]
final_test_probs = meta_model.predict_proba(X_meta_test)[:, 1]


Training Meta-Model (Logistic Regression)...


In [13]:
print("Optimizing threshold for F1-Score...")
best_f1 = 0
best_threshold = 0.5
for t in np.linspace(0.01, 0.99, 200):
    score = f1_score(y, (final_oof_probs > t).astype(int))
    if score > best_f1:
        best_f1 = score
        best_threshold = t

print(f"\n[RESULTS]")
print(f"Optimal Out-of-Fold F1-Score: {best_f1:.6f}")
print(f"Optimal Threshold: {best_threshold:.4f}")

Optimizing threshold for F1-Score...

[RESULTS]
Optimal Out-of-Fold F1-Score: 0.988233
Optimal Threshold: 0.2661


In [14]:
final_labels = (final_test_probs > best_threshold).astype(int)

submission = pd.DataFrame({
    "ID": test_ids,
    "Class": final_labels
})

submission.to_csv("final.csv", index=False)
print("\nfinal.csv generated successfully! Ready for submission.")


final.csv generated successfully! Ready for submission.
